# Code

In [1]:
# ============================================================
# Imports
# ============================================================
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.components.containers import NormalizedLandmark
import time
import math

In [2]:
# ============================================================
# CONFIG
# ============================================================
MODEL_PATH = 'face_landmarker.task'

from dataclasses import dataclass

@dataclass
class EyeConfig:
    iris_ring:     list[int]         # 4 pontos do anel da íris
    iris_center:   int               # 1 ponto central
    ear_points:    list[int]         # 6 pontos pro cálculo de EAR
    corners:       tuple[int,int]    # (externo, interno)

LEFT_EYE  = EyeConfig(
    iris_ring   = [474, 475, 476, 477],
    iris_center = 473,
    ear_points  = [362, 385, 387, 263, 373, 380],
    corners     = (362, 263),
)

RIGHT_EYE = EyeConfig(
    iris_ring   = [469, 470, 471, 472],
    iris_center = 468,
    ear_points  = [33, 160, 158, 133, 153, 144],
    corners     = (33, 133),
)

EYES = [LEFT_EYE, RIGHT_EYE]   # já pronto pro "para each eye in EYES" de antes

# separado, porque não é "por olho" — é config geral do pipeline:
EAR_BLINK_THRESHOLD = 0.21
EAR_CONSEC_FRAMES = 2
SMOOTHING_ALPHA = 0.4
# WINK_TOLERANCE_FRAMES: # TODO: ainda em aberto:


In [3]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def get_point(landmarks: list[NormalizedLandmark], idx: int, w: float, h:float) -> tuple[float, float]:
    lm = landmarks[idx]
    return (lm.x * w, lm.y * h)

# p1, p2: tuplas (x, y) — acesso por índice: p1[0] = x, p1[1] = y
def euclidean(p1: tuple[float, float], p2: tuple[float, float]) -> float:
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def calc_ear(landmarks: list[NormalizedLandmark], eye: EyeConfig, w: float, h: float) -> float:
    points = [get_point(landmarks, idx, w, h) for idx in eye.ear_points]
    vertical1 = euclidean(points[1], points[5])
    vertical2 = euclidean(points[2], points[4])
    horizontal = 2 * euclidean(points[0], points[3])
    if horizontal == 0:
        return 0.0
    ear = (vertical1 + vertical2) / horizontal
    return ear

def is_eye_closed(ear: float) -> bool:
    return ear < EAR_BLINK_THRESHOLD

class EyeBlinkDebouncer:
    def __init__(self, required_consec_frames: int) -> None:
        self._required_consec_frames: int = required_consec_frames
        self._consecutive_closed_frames: int = 0

    def update(self, ear: float) -> bool:
        closed_now = is_eye_closed(ear)
        if closed_now:
            self._consecutive_closed_frames += 1
            return False
        else:
            confirmed = self._consecutive_closed_frames >= self._required_consec_frames
            self._consecutive_closed_frames = 0
            return confirmed

# As its using the debouncer, its not able to detect if the eyes are open.
# In order to be classify_eye_state, it should get above. Maybe change in future.
def classify_blink_wink(left_debouncer: EyeBlinkDebouncer, right_debouncer: EyeBlinkDebouncer, left_ear: float, right_ear: float) -> str | None:
    left_confirmed = left_debouncer.update(left_ear)
    right_confirmed = right_debouncer.update(right_ear)
    match (left_confirmed, right_confirmed):
        case (True, True):      return "blinking"
        case (True, False):     return "left_winking"
        case (False, True):     return "right_winking"
        case (False, False):    return None

def iris_center(landmarks: list[NormalizedLandmark], eye: EyeConfig, w: float, h: float) -> tuple[float, float]:
    pts = [get_point(landmarks, i, w, h) for i in eye.iris_ring + [eye.iris_center]]
    x = sum(p[0] for p in pts) / len(pts)
    y = sum(p[1] for p in pts) / len(pts)
    return (x, y)

def gaze_ratio(iris_pos: tuple[float, float], landmarks: list[NormalizedLandmark], eye: EyeConfig, w: float, h: float) -> float:
    outer = get_point(landmarks, eye.corners[0], w, h)
    inner = get_point(landmarks, eye.corners[1], w, h)
    eye_width = euclidean(outer, inner)
    if eye_width == 0:
        return 0.5
    dist_from_outer = euclidean(iris_pos, outer)
    return dist_from_outer / eye_width  # ~0.0 a 1.0

class ExponentialSmoother:
    def __init__(self, alpha=SMOOTHING_ALPHA):
        self.alpha = alpha
        self.value = None

    def update(self, new_value):
        if self.value is None:
            self.value = new_value
        else:
            self.value = (
                self.alpha * new_value[0] + (1 - self.alpha) * self.value[0],
                self.alpha * new_value[1] + (1 - self.alpha) * self.value[1],
            )
        return self.value


In [4]:
# ============================================================
# SETUP
# ============================================================
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    print("Error: Could not open the camera.")
    exit()

left_iris_smoother = ExponentialSmoother() # É preciso? Porque temos iris_center()
right_iris_smoother = ExponentialSmoother()

left_debouncer  = EyeBlinkDebouncer(required_consec_frames=EAR_CONSEC_FRAMES)
right_debouncer = EyeBlinkDebouncer(required_consec_frames=EAR_CONSEC_FRAMES)

# JUST FOR NOW
blink_total = 0
left_wink_total = 0
right_wink_total = 0

print("Pressione ESC para sair.")

Pressione ESC para sair.


In [5]:
# ============================================================
# LOOP
# ============================================================

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    ts = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image, ts)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        # ---- ÍRIS (posição suavizada) ----
        left_iris_curr = iris_center(landmarks, LEFT_EYE, w, h)
        right_iris_curr = iris_center(landmarks, RIGHT_EYE, w, h)
        left_pos = left_iris_smoother.update(left_iris_curr)
        right_pos = right_iris_smoother.update(right_iris_curr)

        cv2.circle(frame, (int(left_pos[0]), int(left_pos[1])), 3, (0, 255, 255), -1)
        cv2.circle(frame, (int(right_pos[0]), int(right_pos[1])), 3, (0, 255, 255), -1)

        # ---- GAZE RATIO (direção do olhar, 0=fora, 1=dentro) ----
        left_gaze = gaze_ratio(left_pos, landmarks, LEFT_EYE, w, h)
        right_gaze = gaze_ratio(right_pos, landmarks, RIGHT_EYE, w, h)
        avg_gaze = (left_gaze + right_gaze) / 2 # Mantive assim, porque só ficaria diferente caso a pessoa seja vesga ou estiver winking.

        # ---- EAR / DETECÇÃO DE PISCADA ----
        left_ear = calc_ear(landmarks, LEFT_EYE, w, h)
        right_ear = calc_ear(landmarks, RIGHT_EYE, w, h)

        classification = classify_blink_wink(left_debouncer, right_debouncer, left_ear, right_ear)

        match classification:
            case ("blinking"): blink_total += 1
            case ("left_winking"): left_wink_total += 1
            case ("right_winking"): right_wink_total +=1
            case (None): pass

        # ---- Debug na tela ----
        cv2.putText(frame, f"Left EAR: {left_ear:.2f}", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Right EAR: {right_ear:.2f}", (20, 60),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Gaze: {avg_gaze:.2f}", (20, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Blinks: {blink_total}", (20, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Left Winks: {left_wink_total}", (20, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"Right Winks: {right_wink_total}", (20, 180),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow('Eye Tracking', frame)
    if cv2.waitKey(5) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()